# UK BBS · Baseline forecasting

**Stage 2 — example notebook.** Each modelling method can have its own notebook,
while using the same data split, `fit/predict`, and evaluation functions.

Run `01_download_and_preprocess.ipynb` first. This example uses the actual
observed counts; **no absent species–square–year is assumed to be a zero**.

## 1. Load processed observations

The notebook works when Jupyter starts from the repository root or `notebooks/`.
The model input is an ordinary pandas DataFrame. Add metadata columns here in
future notebooks (taking care to use only information available at prediction time).

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "ukbbs" / "forecasting.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the ukbbs-workflow repository")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ukbbs import ForecastTask, LastValueForecaster, chronological_split, run_forecast, save_forecast_results

csv_path = PROJECT_ROOT / "data" / "processed" / "ukbbs_model_data.csv"
if not csv_path.exists():
    raise FileNotFoundError("Run notebook 01 to create data/processed/ukbbs_model_data.csv")
observations = pd.read_csv(csv_path, dtype={"OS_GRID_REF": "string", "species": "string"})
task = ForecastTask()
# Example species; choose another code to experiment with another species.
SPECIES = "CU"
observations = observations.loc[observations["species"].eq(SPECIES)].copy()
if observations.empty:
    raise ValueError(f"No observations for species {SPECIES!r}; select an available species")
display(observations)

SyntaxError: unterminated f-string literal (detected at line 26) (783063015.py, line 26)

## 2. Time-based holdout

Training includes years up to and including the cutoff; evaluation uses later
observations. 2020 remains absent — it is **not** treated as an adjacent year.
This naive *per-series* model cannot forecast squares without training history,
so we explicitly evaluate only previously seen squares.

In [ ]:
train, test = chronological_split(observations, train_through=2018, task=task)
seen_groups = train[list(task.group_cols)].drop_duplicates()
test = test.merge(seen_groups, on=list(task.group_cols), how="inner", validate="many_to_one")
if test.empty:
    raise ValueError("No test groups have training history; choose another cutoff/species")
print(f"Training rows: {len(train):,}; held-out observed rows: {len(test):,}")
print(f"Training years: {train.year.min()}–{train.year.max()}")
print(f"Test years: {test.year.min()}–{test.year.max()}")

## 3. Fit, predict and score using the shared interface

Swap `LastValueForecaster()` for any class implementing `fit(train, task)` and
`predict(future)`. The runner removes test labels before invoking `predict`.
It returns keyed predictions, per-row errors, and MAE/RMSE/bias.

In [ ]:
model = LastValueForecaster()
report = run_forecast(model, train, test, task)
display(pd.Series(report.metrics, name="value").to_frame())
display(report.predictions.head(10))

## 4. Save predictions by model and species

The model name, parameters, task, and run configuration define a **stable hash**.
Each species gets its own CSV in that experiment directory. The root `index.json`
maps human-readable model names to hashes. Set `run_config` to include the data
release, training cutoff, and other settings that distinguish your experiments.


In [ ]:
saved_folder = save_forecast_results(
    report,
    output_root=PROJECT_ROOT / "data" / "predictions",
    model_name="last_value",
    model_params={},
    run_config={"dataset_file": csv_path.name, "train_through": 2018,
                "test_through": int(test.year.max())},
    task=task,
)
print(f"Saved species predictions in {saved_folder}")
print(f"Model lookup: {saved_folder.parent / 'index.json'}")
